# Ranking figures

The **Random Forest** (the primary ranker), the **composite** rules, and the **RF-vs-composite**
comparison behind the consensus shortlist. Split out of `evaluation_plots.ipynb` (single-metric
benchmark) and `composite_plot.ipynb` (building and exploring the composite), which now keep to
their own subjects.

Every figure here reads a pipeline output -- nothing is recomputed, so the figures and the
reported numbers cannot drift apart. Run the pipeline first:

    DATASET=rf ./run_all.sh
    DATASET=rf PYTHONPATH=. python run_rf_importance.py   # optional: the importance comparison


In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from config.datasets import cfg
from config.paths import EVALUATION_DIR, FIGURES_DIR
from analysis.ranking import (plot_rf_performance, plot_rf_importances, plot_rf_importance_comparison,
                              plot_design_scores, plot_ranking_curves, plot_fold_pr_auc,
                              plot_pooled_vs_fold_mean, plot_pooled_vs_per_source,
                              plot_per_source_comparison, plot_standardisation_sensitivity)
from config.palette import apply_plot_style
apply_plot_style()

# Auto-save every figure below to data/figures/<dataset>/ at print resolution. Comment out for
# display only; an explicit save_path= always wins.
from analysis.figures import set_figure_dir
set_figure_dir(FIGURES_DIR / cfg.name)

eval_dir = out = EVALUATION_DIR / cfg.name          # 'out' is the name the composite cells use
rank_dir, cons_dir = eval_dir / "ranking", eval_dir / "consensus"

# rf_cv_metrics.csv is read two ways below: as a plain table (to display) and indexed by `metric`
# (to look single values up). Keep both, under different names.
rf_cv_table = pd.read_csv(rank_dir / "rf_cv_metrics.csv")
rf_cv_indexed = rf_cv_table.set_index("metric")
summary_cv = pd.read_csv(out / "composite" / "composite_cv_eval.csv")
print(f"dataset: {cfg.name}   outputs: {out}")


## RF performance (out-of-fold)

PR and ROC curves over the POOLED out-of-fold predictions (`ranking/rf_oof_scores.csv`): every
point was predicted by a model that never saw it, folds grouped by parent lineage.

Two panels, never one axis -- the PR curve's no-skill line is the prevalence (0.48) and the ROC
curve's is the diagonal. The AUCs in the legends are read from `rf_cv_metrics.csv`, so the curves
and the reported numbers are the same quantity. The rest of that table (precision@10%, the fold
means and PR-AUC enrichment) is shown below the figure.

In [ ]:
rank_dir = eval_dir / "ranking"
if (rank_dir / "rf_oof_scores.csv").exists():
    rf_oof = pd.read_csv(rank_dir / "rf_oof_scores.csv")
    rf_cv = pd.read_csv(rank_dir / "rf_cv_metrics.csv") if (rank_dir / "rf_cv_metrics.csv").exists() else None
    plot_rf_performance(rf_oof, cv_metrics=rf_cv, show=True)
    if rf_cv is not None:
        display(rf_cv[["metric", "value", "std", "n", "n_splits"]])
else:
    print("No rf_oof_scores.csv -- re-run: DATASET=rf PYTHONPATH=. python run_ranking_rf.py")

## RF feature importances

Impurity (Gini) importances of the final Random Forest, written by the RF stage to
`ranking/rf_importances.csv`. Sorted most-important first; the dashed line is the equal share
`1 / n_features`, i.e. what every feature would get if none mattered more.

Impurity importance is computed on the TRAINING data and is biased toward correlated features.
With ~46 correlated confidence metrics it says what the forest *used*, not what predicts binding --
`run_rf_importance.py` computes permutation importance as the cross-check (next section).

In [ ]:
p = eval_dir / "ranking" / "rf_importances.csv"
if p.exists():
    rf_imp = pd.read_csv(p)
    plot_rf_importances(rf_imp, top_n=15, show=True)
else:
    print("No rf_importances.csv -- run the RF ranking stage first.")

## RF importance: Gini vs permutation

`run_rf_importance.py` writes `ranking/rf_importance_comparison.csv` -- three importances for the
SAME features and the SAME model/seed, so the columns differ only by MEASUREMENT method:

| column | model | scored on |
|---|---|---|
| `gini_final` | final (all labelled rows) | structural, read off the trees -- in-sample |
| `perm_final` | final (all labelled rows) | the same rows it was fit on -- in-sample |
| `perm_cv` | grouped-CV train folds | the HELD-OUT fold -- the only out-of-sample column |

Gini and permutation disagree even on the same fitted model and the same rows: with ~46 correlated
confidence metrics, which member of a correlated group wins a split is close to a lottery, so Gini
rewards the winner while permutation reads ~0 because a substitute covers for it. Each panel keeps
its own x-axis -- Gini sums to 1, the permutation columns are in average-precision units.

In [ ]:
p = eval_dir / "ranking" / "rf_importance_comparison.csv"
if p.exists():
    rf_cmp = pd.read_csv(p)
    plot_rf_importance_comparison(rf_cmp, top_n=15, show=True)
else:
    print("No rf_importance_comparison.csv -- run: DATASET=rf PYTHONPATH=. python run_rf_importance.py")

## Design scores: where the shortlist sits

`ranking/rf_design_scores.csv` and `consensus/consensus_scores.csv` score every design; the rug
under each histogram marks `consensus/shortlist.csv`, split by its `consensus` column:

- **consensus** -- top-k on BOTH the RF and the composite
- **primary_only** -- top-k on the RF alone

One shared bin grid per figure, colour per design set from `palette.DATASET`. The RF threshold is
the one used to pick candidates; the composite has no cutoff, so that panel has no line.

Read the two together: on the RF axis the shortlist is one tight cluster (it *is* the RF top-k),
while on the composite axis the two groups separate -- which is what the rank agreement in
`consensus/agreement.txt` looks like design by design.

In [ ]:
rank_dir, cons_dir = eval_dir / "ranking", eval_dir / "consensus"
if (cons_dir / "consensus_scores.csv").exists():
    design_scores = pd.read_csv(cons_dir / "consensus_scores.csv")
    if (rank_dir / "rf_design_scores.csv").exists():   # consensus_scores.csv carries no 'dataset'
        design_scores = design_scores.merge(
            pd.read_csv(rank_dir / "rf_design_scores.csv")[["sample", "dataset"]], on="sample", how="left")
    shortlist = pd.read_csv(cons_dir / "shortlist.csv")
    picks = ({label: g["sample"] for label, g in shortlist.groupby("consensus")}
             if "consensus" in shortlist.columns else shortlist["sample"])

    # the score distributions on their own ...
    plot_design_scores(design_scores, score="rf_score", threshold=0.63,
                       xlabel="RF p(binder)", show=True)
    plot_design_scores(design_scores, score="comp_score",
                       xlabel="Composite (product) score", show=True)
    # ... and with the shortlist overlaid (saved under a separate *_with_shortlist name)
    plot_design_scores(design_scores, score="rf_score", threshold=0.63,
                       xlabel="RF p(binder)", selected=picks, show=True)
    plot_design_scores(design_scores, score="comp_score",
                       xlabel="Composite (product) score", selected=picks, show=True)
    display(shortlist.groupby("consensus")[["rf_score", "comp_score"]].agg(["min", "max", "count"]).round(3))
else:
    print("No consensus_scores.csv -- run run_consensus.py first.")

## RF vs composite

The two rankers side by side. Pooled and per-source measures rank them **oppositely** on this
benchmark, so both views are shown -- neither alone is a fair summary.


In [ ]:
# RF vs ONE composite rule throughout this section. The three rules score within 0.01 PR-AUC of
# each other (0.548 / 0.546 / 0.538), so their curves overlap -- plotting all three adds no signal.
composite_model = "composite_product"

# 1. pooled PR and ROC curves, from the per-sample OUT-OF-FOLD predictions both stages now write.
rf_oof = (pd.read_csv(out / "ranking" / "rf_oof_scores.csv")
          .rename(columns={"p_binder_oof": "score"}).assign(model="Random Forest"))
comp_oof = pd.read_csv(out / "composite" / "composite_oof_scores.csv")
comp_oof = comp_oof[comp_oof["model"] == composite_model].assign(model=composite_model.replace("_", " "))
oof = pd.concat([rf_oof[["model", "binder", "score"]], comp_oof[["model", "binder", "score"]]],
                ignore_index=True)
plot_ranking_curves(oof, show=True)

# 2. per-fold PR-AUC, against each fold's prevalence baseline. Grouped folds are close to
#    leave-one-source-out, so the spread IS cross-source transfer, not sampling noise.
by_fold = pd.concat([pd.read_csv(out / "composite" / "cv_by_fold.csv"),
                     pd.read_csv(out / "ranking" / "cv_by_fold.csv")], ignore_index=True)
plot_fold_pr_auc(by_fold[by_fold["model"].isin(["rf", composite_model])], show=True)

# 3. the two estimators of PR-AUC. Pooling weights folds by their positives, the fold-mean weights
#    them equally, so the two can disagree about the winner.
summary = pd.DataFrame(
    [{"model": "rf",
      "pr_auc_pooled": rf_cv_indexed.loc["pr_auc_oof", "value"],
      "pr_auc_fold_mean": rf_cv_indexed.loc["pr_auc_fold_mean", "value"],
      "pr_auc_fold_std": rf_cv_indexed.loc["pr_auc_fold_mean", "std"]}]
    + [{"model": r["model"], "pr_auc_pooled": r["pr_auc"],
        "pr_auc_fold_mean": r["pr_auc_fold_mean"], "pr_auc_fold_std": r["pr_auc_fold_std"]}
       for _, r in summary_cv[summary_cv["model"] == composite_model].iterrows()])
plot_pooled_vs_fold_mean(summary, show=True);

# 3b. the three combining rules against each other: mean, PR-AUC-weighted mean and the geometric
#     mean of Phi(z) score within 0.01 PR-AUC of one another, so the curves sit on top of each
#     other. This is the evidence for reporting ONE rule above -- the metric set drives the
#     composite's behaviour, not the choice of rule.
rules = ["composite_consensus", "composite_weighted", "composite_product"]
rule_oof = pd.read_csv(out / "composite" / "composite_oof_scores.csv")
rule_oof = rule_oof[rule_oof["model"].isin(rules)].copy()
rule_oof["model"] = rule_oof["model"].str.replace("composite_", "composite ")
plot_ranking_curves(rule_oof[["model", "binder", "score"]], show=True)

In [ ]:
# 4. RF vs the composite on BOTH views: the pooled and per-source measures rank them oppositely
#    here, so neither alone is a fair summary.
plot_pooled_vs_per_source(rf_cv_indexed.reset_index(), summary_cv, composite_model=composite_model, show=True)

# 5. where each method wins: the per-source values the aggregates above are computed from.
variants_dir = out / "composite" / "variants"
rf_src = pd.read_csv(out / "ranking" / "rf_per_source.csv").assign(model="Random Forest")
per_source = rf_src[["source", "model", "ap_norm"]]
if (variants_dir / "variant_per_source.csv").exists():
    vc = pd.read_csv(variants_dir / "variant_comparison.csv")
    rule = composite_model.replace("composite_", "")
    target = float(summary_cv.set_index("model").loc[composite_model, "ap_norm_ds_mean"])
    match = vc[(vc["rule"] == rule) & (vc["ap_norm_ds_mean"].sub(target).abs() < 1e-6)]
    if len(match):
        variant = match["variant"].iloc[0]
        print(f"reported {composite_model} == variant {variant}")
        vps = pd.read_csv(variants_dir / "variant_per_source.csv")
        sel = vps[(vps["variant"] == variant) & (vps["rule"] == rule)].assign(model=composite_model.replace("_", " "))
        per_source = pd.concat([per_source, sel[["source", "model", "ap_norm"]]], ignore_index=True)
plot_per_source_comparison(per_source, metric="ap_norm", show=True)

# 6. how much the composite moves when the standardisation changes
sens = variants_dir / "standardisation_sensitivity.csv"
if sens.exists():
    plot_standardisation_sensitivity(pd.read_csv(sens), show=True)